# CMU 11-768 Assignment 3 · ScottyRL project walkthrough

In Assignment 1, the model used tools inside an agent harness. Here the environment scores the agent's actions, and the trainer uses those scores to change the policy.

Open this notebook in Colab, choose **Runtime → Change runtime type → T4 GPU**, and run the cells in order. With a CPU runtime, the offline exercises, small update, and log analysis still run. The optional Qwen inference cell uses a GPU and downloads public model weights. The paid Modal commands are in Markdown cells for use in Codespaces.

| Section | Follow in the repository | What you can run |
| --- | --- | --- |
| Parts 0–1 | `rollout.py` → `advantages.py` → `datagen.py` | Number Search, TextCraft tools, and the training masks |
| Parts 2–3 | `train/common.py`, `sync.py`, `tinker_adapter.py`, `async_grpo.py` | A small Torch update on the T4 and both trainer loops with an offline stand-in for Tinker |
| Parts 4–5 | `configs/`, `runs/`, `docs/Report.md` | Compare real Modal logs and optionally sample an actual Qwen model on the T4 |
| Modal lab | `skyrl_tinker_modal/`, `docs/Runbook.md` | Commands for the two- and four-H100 deployments used in the project |

## 0 · Set up the source checkout

The examples pin the assignment source and run logs to the commit that includes the final report. Colab's PyTorch installation provides CUDA. The assignment's own tests run in a separate `uv` environment.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

SOURCE_COMMIT = "477a42d40993b18e22f06f50b91f729176873ab6"
repo = Path.cwd() / "assignment3_source"

if not repo.exists():
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/xqzr2000/cmu-11-768-assignment-3.git", str(repo)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
if importlib.util.find_spec("yaml") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "pyyaml"], check=True)
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
sys.path.insert(0, str(repo))

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

## 1 · Run a rollout and turn it into training data

### 1.1 · Follow the same agent loop as the training run

`XMLToolCallAgent` renders the prompt and asks the policy for a completion. `parse_tool_call` converts the completion into a tool name and literal arguments. `run_episode` feeds the parsed call to the environment and stores the observation. Here, `MockPolicy` gives the real agent two fixed replies so we can follow every step.

In [ ]:
from scottyrl.agents.xml_tool_call import XMLToolCallAgent, parse_tool_call
from scottyrl.envs.number_search import NumberSearchEnv
from scottyrl.policy import MockPolicy
from scottyrl.rollout import PENDING_TRAJECTORIES, run_episode
from scottyrl.types import load_tasks

task = load_tasks(repo / "data" / "number_search_train.jsonl")[0]
policy = MockPolicy(["<tool>guess(100)</tool>", "<tool>guess(228)</tool>"])
trajectory = await run_episode(XMLToolCallAgent(policy), NumberSearchEnv(task))

for turn in PENDING_TRAJECTORIES[-1][-1]:
    print(f"{turn['agent']} -> {turn['env']}  (reward={turn['reward']:.0f})")
assert trajectory.reward == 1 and trajectory.num_steps == 2

Try changing the second guess. A wrong guess produces another observation, and the loop continues until the environment reaches its step limit. The TextCraft environment uses the same loop. Its reward depends on the final inventory and the `finish()` call. The following episode stops before crafting anything.

In [ ]:
from scottyrl.envs.textcraft.env import TextCraftEnv

craft_task = load_tasks(repo / "data" / "textcraft_val.jsonl")[0]
craft_env = TextCraftEnv(craft_task)
print((await craft_env.reset()).text)
target = next(iter(craft_task.info["target_items"]))
print((await craft_env.step(parse_tool_call(f"<tool>get_info(['{target}'])</tool>"))).text)
finish = await craft_env.step(parse_tool_call('<tool>finish("done")</tool>'))
assert finish.done and finish.reward == 0
print(finish.text, "Reward:", finish.reward)

### 1.2 · Turn episode rewards into advantages

REINFORCE in this assignment uses $A_i = R_i$. The assignment's GRPO variant uses $A_i = R_i - overline{R}$, where the mean is over attempts at the **same** task. The winning episode in a group of one win and three losses gets $+0.75$. The others get $-0.25$ each. Equal rewards give a zero vector, so the implementation signals that the group can be skipped.

In [ ]:
from scottyrl.advantages import compute_group_advantages

rewards = [1, 0, 0, 0]
print("REINFORCE", compute_group_advantages(rewards, method="reinforce"))
print("GRPO", compute_group_advantages(rewards, method="grpo"))
print("All-success GRPO group", compute_group_advantages([1, 1, 1, 1]))
assert compute_group_advantages(rewards) == [0.75, -0.25, -0.25, -0.25]
assert compute_group_advantages([1, 1, 1, 1]) is None

### 1.3 · Stitch two calls and inspect the next-token shift

The second prompt contains the first prompt, its completion, and one new observation token. `stitch_trajectory` reuses the prefix. At position $j$, the input is `tokens[j]` and the target is `tokens[j+1]`. The observation token is still context for the next prediction, with zero contribution to the loss. When the prompt rewrites history, the stitcher starts a fresh sequence and preserves each call's actual context.

In [ ]:
from scottyrl.datagen import stitch_trajectory, trajectory_to_training_sequences
from scottyrl.types import LLMCall, Trajectory

first = LLMCall([10, 11], [20, 21], [-0.2, -0.3])
second = LLMCall([10, 11, 20, 21, 30], [40], [-0.4])
example = Trajectory(task=task, calls=[first, second], reward=1.0, num_steps=2)
advantage = compute_group_advantages(rewards)[0]
(sequence,) = trajectory_to_training_sequences(example, advantage)
print(" input  target  mask   log p      A")
for inputs in zip(sequence.tokens[:-1], sequence.tokens[1:], sequence.mask,
                  sequence.logprobs, sequence.advantages):
    token, target, mask, logprob, credit = inputs
    print(f"{token:6d}{target:8d}{mask:6d}{logprob:8.1f}{credit:7.2f}")
assert sequence.mask == [0, 1, 1, 0, 1]

rewritten = LLMCall([10, 11, 30], [40], [-0.4])
split = stitch_trajectory(Trajectory(task, [first, rewritten], 1.0, 2))
assert len(split) == 2
print("After a prefix break:", [part.tokens for part in split])

## 2 · Run one synchronous policy update on the T4

A full LLM chooses every token of a tool call. To see the gradient on a T4 in seconds, this exercise represents each of two complete guesses as a single action token. A trainable pair of logits chooses between the guesses. The real ScottyRL agent parses the XML, Number Search scores each episode, and `groups_to_batch` supplies the mask, sampled log probabilities, and centered advantages used by the update.

For an action token with sampling log probability $ell_{mathrm{old}}$ and current log probability $ell_	heta$, the ratio is $r = exp(ell_	heta - ell_{mathrm{old}})$. The small Torch implementation minimizes the negative mean of $min(rA, operatorname{clip}(r,1-epsilon,1+epsilon)A)$ over action tokens. Sampling and updating start with the same policy, so the initial ratios should be 1. A later pass over the same data uses the saved log probabilities and can give different ratios. The exercise uses one task and a one-turn budget so you can inspect the direction of the update.

In [ ]:
from scottyrl.policy import Completion
from scottyrl.train.common import collect_group, groups_to_batch
from scottyrl.types import Task


class TwoGuessPolicy(MockPolicy):
    first_action_token = 256  # MockPolicy uses 0–255 for prompt bytes.
    guesses = (1, 2)

    def __init__(self, logits):
        self.logits = logits

    async def complete(self, prompt_tokens):
        with torch.no_grad():
            distribution = torch.distributions.Categorical(logits=self.logits)
            action = distribution.sample()
            index = action.item()
            return Completion(
                text=f"<tool>guess({self.guesses[index]})</tool>",
                tokens=[self.first_action_token + index],
                logprobs=[distribution.log_prob(action).item()],
            )


torch.manual_seed(11768)
logits = torch.nn.Parameter(torch.zeros(2, device=device))
optimizer = torch.optim.Adam([logits], lr=0.1)
toy_task = Task(
    id="toy-number-search", goal="Guess the hidden number in [1, 2].",
    max_steps=1, info={"low": 1, "high": 2, "target": 2},
)

policy = TwoGuessPolicy(logits)
group = await collect_group(policy, toy_task, NumberSearchEnv, group_size=32)
batch, stats = groups_to_batch([group], method="grpo")
print("Wins:", sum(episode.reward for episode in group), "/", len(group))
print("Batch stats:", stats)
assert 0 < stats["mean_reward"] < 1, "A mixed group is needed for the demonstration."

In [ ]:
actions, old_logprobs, advantages = zip(*[
    (token - policy.first_action_token, logprob, advantage)
    for sequence in batch
    for token, mask, logprob, advantage in zip(
        sequence.tokens[1:], sequence.mask, sequence.logprobs, sequence.advantages
    )
    if mask
])
actions = torch.tensor(actions, device=device, dtype=torch.long)
old_logprobs = torch.tensor(old_logprobs, device=device)
advantages = torch.tensor(advantages, device=device)

current_logprobs = logits.log_softmax(dim=0)[actions]
ratio = (current_logprobs - old_logprobs).exp()
assert torch.allclose(ratio, torch.ones_like(ratio), atol=1e-6)
print("Current/sample ratio before update:", ratio.mean().item())

before = logits.softmax(dim=0).detach()
optimizer.zero_grad()
loss = -torch.minimum(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages).mean()
loss.backward()
optimizer.step()
after = logits.softmax(dim=0).detach()
print(f"P(guess 2): {before[1].item():.3f} -> {after[1].item():.3f}")
updated_ratios = (logits.detach().log_softmax(dim=0)[actions] - old_logprobs).exp()
print("Ratios for both actions on the same sampled data:", updated_ratios.unique().tolist())
assert after[1] > before[1]

The probability of guessing 2 increases after the update. The objective value before updating can be near zero because the GRPO advantages sum to zero, while the gradients on the two actions point in opposite directions.

In the H100 runs, `SyncTrainer` samples whole episodes, assembles the token batch, and ships it to `TinkerSession.train_step`. That method converts the sequences to Tinker datums and scales advantages by the total action-token count. `submit_update` queues `forward_backward_async` and `optim_step_async` before waiting for results. SkyRL computes on the LoRA weights, then ScottyRL publishes a new sampling snapshot for the next batch. The two-guess policy above makes the objective visible without loading the 4B checkpoint.

## 3 · Compare synchronous and asynchronous trainers

`SyncTrainer` finishes a rollout batch before calling the learner. In `AsyncGRPOTrainer`, workers each hold a sampler snapshot and put finished groups in a bounded queue. The learner discards groups when `current_version - sampled_version > max_staleness`. For example, version 5 data is accepted at version 7 with a limit of 2, and dropped at version 8. PPO clips the ratio for accepted groups. The no-weighting ablation uses advantage-weighted cross-entropy and still logs the ratios for comparison.

Run the project's end-to-end trainer tests. They use real Number Search tasks and the real agent, queue, and batch code, with scripted samples and a fake Tinker session that records updates and snapshots.

In [ ]:
subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.12"],
    cwd=repo,
    check=True,
)
subprocess.run(
    ["uv", "run", "--locked", "--no-sync", "pytest", "-q",
     "tests/test_part1_pipeline.py", "tests/test_offline_end_to_end.py"],
    cwd=repo,
    check=True,
)

## 4 · Compare the Number Search runs using the committed logs

The first plot uses training steps on its horizontal axis. The second uses minutes since the step-0 evaluation, so you can see the effect of overlapping rollouts with updates. In the async trainers, the evaluation policy version belongs to the sampled snapshot, while `time` is when its record was logged. Cold starts before the first evaluation are outside this comparison.

The table below the plots also shows the largest per-step importance-ratio standard deviation for the three async runs. A final score near 99% tells little about the spread of token-wise ratios earlier in training.

In [ ]:
import json

import matplotlib.pyplot as plt


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


runs_dir = repo / "runs"
number_search_runs = {
    "Sync REINFORCE": "p2a_number_search_reinforce",
    "Sync GRPO G=8": "p2b_number_search_grpo",
    "Sync GRPO G=2": "p41_number_search_grpo_g2",
    "Async GRPO": "p3_number_search_async",
    "Async staleness 4, IW": "p43_number_search_async_stale4_iw",
    "Async staleness 4, no IW": "p43_number_search_async_stale4_noiw",
}
fig, (by_step, by_time) = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for label, stem in number_search_runs.items():
    records = read_jsonl(runs_dir / f"{stem}.jsonl")
    evaluations = sorted(
        (row for row in records if "eval/success_rate" in row),
        key=lambda row: row["step"],
    )
    success = [row["eval/success_rate"] for row in evaluations]
    by_step.plot([row["step"] for row in evaluations], success, marker="o", label=label)
    if label in ("Sync GRPO G=8", "Async GRPO"):
        start = evaluations[0]["time"]
        by_time.plot([(row["time"] - start) / 60 for row in evaluations],
                     success, marker="o", label=label)

by_step.set(xlabel="Training step", ylabel="Held-out success")
by_time.set(xlabel="Minutes since step-0 evaluation", ylabel="Held-out success")
by_step.legend(fontsize="small")
by_time.legend(fontsize="small")
plt.show()

for label, stem in number_search_runs.items():
    if label.startswith("Async"):
        rows = read_jsonl(runs_dir / f"{stem}.jsonl")
        spread = max(row["train/importance_weight_std"] for row in rows
                     if "train/importance_weight_std" in row)
        print(f"{label:27s}  largest ratio std = {spread:.3f}")

With binary rewards and probability of success $p$, a group of size $G$ has no GRPO signal with probability $p^G+(1-p)^G$ if attempts are independent with that success probability. At $p=0.35$, this is about 55% for $G=2$ and about 3.2% for $G=8$. At $p=0.99$, most groups of eight are all-success. Use the logged diagnostics to see when the early small-group problem becomes a late saturated-policy problem.

In [ ]:
for label in ("Sync GRPO G=2", "Sync GRPO G=8"):
    stem = number_search_runs[label]
    rows = [row for row in read_jsonl(runs_dir / f"{stem}.jsonl")
            if "train/frac_degenerate_groups" in row]
    for window in (rows[:10], rows[-10:]):
        degenerate = sum(row["train/frac_degenerate_groups"] for row in window) / len(window)
        print(f"{label}, steps {window[0]['step']}-{window[-1]['step']}: {degenerate:.1%}")

## 5 · Read the TextCraft evaluation

The detached H100 training run left a step-41 adapter. The standalone vLLM server exposes the base model and the same model with the adapter under the alias `a3`. Both evaluation files cover four attempts on each of the same 50 validation tasks. **Success rate** uses all 200 attempts. **pass@4** counts the fraction of tasks with at least one successful attempt.

Recompute both metrics using the per-episode JSONL files. The course's harder validation split was outside this reduced evaluation.

In [ ]:
from collections import defaultdict
from statistics import mean

for model_file in ("p5_eval_Qwen_Qwen3.5-4B", "p5_eval_a3"):
    attempts = read_jsonl(runs_dir / f"{model_file}.jsonl")
    summary = json.loads((runs_dir / f"{model_file}.json").read_text())
    by_task = defaultdict(list)
    for row in attempts:
        by_task[row["task_id"]].append(row["reward"])

    assert len(by_task) == 50 and all(len(group) == 4 for group in by_task.values())
    success = mean(row["reward"] for row in attempts)
    pass_at_4 = mean(max(group) for group in by_task.values())
    assert success == summary["success_rate"] and pass_at_4 == summary["pass@4"]
    print(f"{summary['model']:18s} success={success:5.1%}  pass@4={pass_at_4:5.1%}"
          f"  mean turns={mean(row['num_steps'] for row in attempts):.1f}")

The adapter finishes more episodes and uses fewer turns. The report compared saved transcripts across training and found that the share of turns with no usable tool call fell from 64% at step 0 to 29% at step 30. It also found no completed task among those requiring more than 31 different crafts. The 75-turn cap and a reward given only for completion make partial progress hard to learn from.

The [final report](https://github.com/xqzr2000/cmu-11-768-assignment-3/blob/main/docs/Report.md) has plots for tool-call usability, task sizes, and the share of groups with a learning signal.

## 6 · Optional T4 inference with Qwen3.5-0.8B

The [0.8B Qwen model](https://huggingface.co/Qwen/Qwen3.5-0.8B) lets a T4 run the real XML agent with model-generated output. This example uses FP16, a short Number Search episode, and a 64-token generation cap. It downloads weights on the first run. Set `RUN_QWEN_DEMO = True` to try it. Watch the raw completion and the parsed call even if the episode fails. The exercise uses a smaller base model and a shorter turn budget than the scored experiments.

In [ ]:
from dataclasses import replace
from importlib.metadata import PackageNotFoundError, version

RUN_QWEN_DEMO = False

if RUN_QWEN_DEMO and device.type == "cuda":
    try:
        transformers_version = version("transformers")
    except PackageNotFoundError:
        transformers_version = None
    if transformers_version != "5.17.0":
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "transformers==5.17.0"],
            check=True,
        )
    from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration
    from scottyrl.policy import ChatTemplatePolicy

    model_id = "Qwen/Qwen3.5-0.8B"
    model_revision = "2fc06364715b967f1860aea9cf38778875588b17"
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        model_id, revision=model_revision, dtype=torch.float16, attn_implementation="sdpa",
    ).to(device).eval()

    class ColabQwenPolicy(ChatTemplatePolicy):
        async def complete(self, prompt_tokens):
            return self.generate_completion(prompt_tokens)

        @torch.inference_mode()
        def generate_completion(self, prompt_tokens):
            input_ids = torch.tensor([prompt_tokens], device=device)
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=torch.ones_like(input_ids),
                max_new_tokens=self.max_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_k=0,
                top_p=1.0,
                eos_token_id=self.stop_token_ids,
                pad_token_id=tokenizer.pad_token_id,
                return_dict_in_generate=True,
                output_scores=True,
            )
            ids = generated.sequences[0, input_ids.shape[1]:]
            logprobs = [
                scores[0].float().log_softmax(dim=-1)[token].item()
                for scores, token in zip(generated.scores, ids)
            ]
            return Completion(
                text=tokenizer.decode(ids.tolist(), skip_special_tokens=False),
                tokens=ids.tolist(),
                logprobs=logprobs,
            )

    chat_template = (repo / "scottyrl" / "chat_templates" / "qwen3_5.jinja").read_text()
    local_policy = ColabQwenPolicy(tokenizer, 64, 1.0, chat_template, enable_thinking=False)
    held_out = load_tasks(repo / "data" / "number_search_val.jsonl")[0]
    short_task = replace(held_out, max_steps=min(3, held_out.max_steps))
    result = await run_episode(XMLToolCallAgent(local_policy), NumberSearchEnv(short_task))

    for turn in PENDING_TRAJECTORIES[-1][-1]:
        print("Completion:", repr(turn["agent"]))
        print("Tool:", turn["tool_call"])
        print("Observation:", turn["env"])
    print(f"Episode reward: {result.reward:.0f} after {result.num_steps} turns")
    del model
    torch.cuda.empty_cache()
else:
    print("To run live Qwen inference, enable RUN_QWEN_DEMO in a GPU runtime.")

## 7 · Modal H100 lab

The assignment's SkyRL deployments provision multiple H100s inside one GPU container. A Colab T4 is a single GPU, so use Modal when reproducing those deployments. The commands below run in a terminal from the repo root unless shown otherwise. They require a Modal workspace and incur usage charges. See the [runbook](https://github.com/xqzr2000/cmu-11-768-assignment-3/blob/main/docs/Runbook.md) for the complete schedule and the [backend guide](https://github.com/xqzr2000/cmu-11-768-assignment-3/blob/main/skyrl_tinker_modal/README.md) for the inherited launch details.

| Experiment | Modal server | GPU roles |
| --- | --- | --- |
| Sync Number Search | `serve_number_search_sync.py` | Two H100s switch between sampling and updates |
| Async Number Search | `serve_number_search_async.py` | One sampler H100 and one learner H100 |
| Async TextCraft | `serve_textcraft.py` | Three sampler H100s and one learner H100 |
| Standalone TextCraft evaluation | `serve_evaluation.py` | Two H100s serving the base and adapter through vLLM |

### 7.1 · A short Number Search probe

Install the Modal extra and authenticate with `uv run modal setup` or your Codespaces Modal secrets. Build and smoke-check the image if this is your first deployment. The training client uses the URL printed by `modal deploy`. The first call can take several minutes while model weights and the backend load.

```bash
uv sync --locked --extra modal
(cd skyrl_tinker_modal && uv run --extra modal modal run serve_number_search_sync.py::smoke_install)
(cd skyrl_tinker_modal && uv run --extra modal modal deploy serve_number_search_sync.py)

read -rsp 'Paste the deployed Tinker URL: ' TINKER_BASE_URL
printf '\n'
export TINKER_BASE_URL

uv run --extra modal python scripts/train_sync.py --config configs/p2b_number_search_grpo.yaml \
  --set train.num_steps=10 --set train.eval_every=2 --set train.eval_tasks=20 \
  --set "train.run_name=probe_walkthrough_$(date +%s)"

uv run --extra modal modal app stop scottyrl-ns-sync
```

Check for a step-0 evaluation, a ratio mean near 1.0, and the final checkpoint manifest before stopping the app. For async Number Search, deploy `serve_number_search_async.py`, point `TINKER_BASE_URL` at its URL, and run `scripts/train_async.py` with `configs/p3_number_search_async.yaml` and a fresh probe name. Stop `scottyrl-ns-async` when that run finishes. Keep the endpoint URL private because it can wake the GPU server.

### 7.2 · Train TextCraft with the client on Modal

Deploy the four-H100 backend and the CPU trainer separately. `launch_on_modal.py` looks up the backend, spawns the CPU job, and returns a call ID. Closing Codespaces after launch leaves the training job running. `modal app logs -f` follows output, and Ctrl-C stops the log stream.

```bash
(cd skyrl_tinker_modal && uv run --extra modal modal deploy serve_textcraft.py)
uv run --extra modal modal deploy skyrl_tinker_modal/run_trainer.py

uv run --extra modal python scripts/launch_on_modal.py --config configs/p5_textcraft_grpo.yaml \
  --set train.num_steps=2 --set train.eval_every=0 \
  --set "train.run_name=probe_p5_walkthrough_$(date +%s)"
uv run --extra modal modal app logs -f scottyrl-trainer
```

The full 50-step config uses 16 tasks × 8 rollouts per step. After a successful probe, launch the full run with its own run name.

```bash
uv run --extra modal python scripts/launch_on_modal.py --config configs/p5_textcraft_grpo.yaml \
  --set train.run_name=p5_walkthrough
uv run --extra modal modal app logs -f scottyrl-trainer
```

Use `--resume` with the same run name and config to continue from persisted state. The trainer volume holds logs. The backend state volume holds optimizer checkpoints and sampler snapshots. After the runner reports completion, download the manifest and stop the GPU app.

```bash
uv run --extra modal modal volume ls scottyrl-trainer-runs runs
uv run --extra modal modal volume get scottyrl-trainer-runs \
  runs/p5_walkthrough.checkpoints.json runs/p5_walkthrough.checkpoints.json
uv run --extra modal modal app stop scottyrl-textcraft
```

### 7.3 · Evaluate a saved adapter

The manifest's `final_sampler_path` identifies the adapter in the training state volume. Its model ID and snapshot name come from your run. Extract that archive into `runs/p5_adapter`. Then deploy vLLM with the adapter and launch the reduced protocol from a Modal CPU container.

```bash
mkdir -p runs/p5_adapter
read -rp 'Model ID from final_sampler_path: ' MODEL_ID
read -rp 'Snapshot name from final_sampler_path: ' SNAPSHOT
uv run --extra modal modal volume get scottyrl-textcraft-state \
  "checkpoints/${MODEL_ID}/sampler_weights/${SNAPSHOT}.tar.gz" runs/p5_adapter.tar.gz
tar -xf runs/p5_adapter.tar.gz -C runs/p5_adapter

(cd skyrl_tinker_modal && LORA_ADAPTER=../runs/p5_adapter uv run --extra modal modal deploy serve_evaluation.py)
uv run --extra modal modal deploy skyrl_tinker_modal/run_trainer.py
uv run --extra modal python scripts/launch_eval_on_modal.py --model a3 \
  --tag "probe_eval_walkthrough_$(date +%s)" --k 1 --num-tasks 2
uv run --extra modal modal app logs -f scottyrl-trainer
```

After the probe finishes, run the two-model evaluation using a fresh tag for the new adapter.

```bash
uv run --extra modal python scripts/launch_eval_on_modal.py \
  --model a3 --model Qwen/Qwen3.5-4B --tag eval_walkthrough --num-tasks 50 --k 4
uv run --extra modal modal app logs -f scottyrl-trainer
```

When both model summaries are written, stop `scottyrl-evaluation`. The results can be downloaded from `scottyrl-trainer-runs`. A directory download takes the parent directory as the destination. Progress lines report request rate, retries, and completed episodes. The evaluation records each finished episode so rerunning with the same tag can continue from it.

### 7.4 · What to look for when a run stalls

| Symptom we saw | How we checked it | Fix used in this repo |
| --- | --- | --- |
| Image builder rejected by the local CLI | The failure happened before any container started | Targeted update of the locked Modal client |
| An empty run after stopping an app | The client had yet to reach step 0 and had no checkpoint | Redeployed and restarted the interrupted run. Later stops waited for the final checkpoint line. |
| Tinker requests stuck for up to 7,200 seconds | Trainer output stopped; the GPU-side cause remained unknown | CPU runner retries from periodic checkpoints and treats an hour without output as a hang |
| Evaluation client stalled with a fast server | Compared server latency, client throughput and active requests | Shorter timeouts, per-episode writes, and evaluation from a Modal CPU container |

Follow the [day-by-day learning log](https://github.com/xqzr2000/cmu-11-768-assignment-3/blob/main/docs/Learning-Progress.md) for the run sequence and the [Modal journal](https://github.com/xqzr2000/cmu-11-768-assignment-3/blob/main/docs/cloud-walker-Modal.md) for the infrastructure details.